# TruthLens AI — Phase 8.3: QLoRA fine-tuning (Colab T4)

Fine-tunes **Qwen2.5-7B-Instruct** on the RAG-format dataset from Phase 8.2
(`data/finetuning/train.jsonl`, `val.jsonl`).

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Design:
- Loss on the **assistant answer only** (prompt + sources masked with -100).
- Same chat template as inference (`app/rag/llm.py` sends one user turn).
- 4-bit NF4 + LoRA r=16 on all attention and MLP projections; fp16 (T4 has no bf16).
- Best checkpoint chosen by validation loss.

Library versions are pinned to match the local inference environment so the adapter loads in Phase 8.4.

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.2 accelerate==0.34.2 bitsandbytes

In [ ]:
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Load the dataset

**Before running this cell:** open the 📁 Files sidebar, click upload, and add `train.jsonl` and
`val.jsonl` from `data\finetuning\`. (Files are wiped whenever the runtime resets.)

In [ ]:
import json
from pathlib import Path
from google.colab import files

missing = [name for name in ("train.jsonl", "val.jsonl") if not Path(name).exists()]
if missing:
    # fail fast: files.upload() blocks silently under "Run all" and can hang in some browsers
    raise FileNotFoundError(
        f"Missing {missing} in /content. Upload them via the Files sidebar, then re-run this cell."
    )

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_rows, val_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl")
count = lambda rows, t: sum(r["meta"]["type"] == t for r in rows)
print(f"train: {len(train_rows)} ({count(train_rows, 'refusal')} refusals)")
print(f"val:   {len(val_rows)} ({count(val_rows, 'refusal')} refusals)")

## 2. Config

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "qwen25-7b-truthlens-lora"
MAX_SEQ_LENGTH = 2048   # Phase 8.2 dropped anything longer; nothing is truncated

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8          # effective batch 8
EVAL_STEPS = 20
SEED = 42

## 3. Tokenize with answer-only labels

The prompt is everything up to and including `<|im_start|>assistant\n`. Its tokens get label -100,
so the loss covers only the cited answer and the `<|im_end|>` stop token.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

def tokenize_example(row):
    messages = row["messages"]
    prompt_ids = tokenizer.apply_chat_template(messages[:1], tokenize=True, add_generation_prompt=True)
    full_ids = tokenizer.apply_chat_template(messages, tokenize=True)
    # if the template ever changes, fail loudly rather than mask the wrong span
    assert full_ids[: len(prompt_ids)] == prompt_ids, "prompt is not a prefix of the full sequence"
    assert len(full_ids) <= MAX_SEQ_LENGTH, f"example too long: {len(full_ids)}"
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids, "labels": labels}

train_ds = [tokenize_example(r) for r in train_rows]
val_ds = [tokenize_example(r) for r in val_rows]

lengths = [len(x["input_ids"]) for x in train_ds]
answer_tokens = [sum(l != -100 for l in x["labels"]) for x in train_ds]
print(f"tokens/example: mean {sum(lengths) / len(lengths):.0f}, max {max(lengths)}")
print(f"answer tokens/example: mean {sum(answer_tokens) / len(answer_tokens):.0f}")
print("\n--- trained span of example 0 ---")
print(tokenizer.decode([t for t in train_ds[0]["labels"] if t != -100]))

In [ ]:
def collate(batch):
    width = max(len(x["input_ids"]) for x in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([x["input_ids"] + [pad] * (width - len(x["input_ids"])) for x in batch]),
        "attention_mask": torch.tensor([[1] * len(x["input_ids"]) + [0] * (width - len(x["input_ids"])) for x in batch]),
        "labels": torch.tensor([x["labels"] + [-100] * (width - len(x["labels"])) for x in batch]),
    }

## 4. Load Qwen in 4-bit and attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"VRAM after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 5. Baseline generations (before training)

One answerable and one refusal example from the validation set, for a before/after comparison.

In [ ]:
def generate_answer(row, max_new_tokens=200):
    prompt = tokenizer.apply_chat_template(row["messages"][:1], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.config.use_cache = True
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    model.config.use_cache = False
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

probe_rows = [
    next(r for r in val_rows if r["meta"]["type"] == "answerable"),
    next(r for r in val_rows if r["meta"]["type"] == "refusal"),
]
before = {}
for r in probe_rows:
    question = r["messages"][0]["content"].split("QUESTION:")[-1].split("ANSWER:")[0].strip()
    before[question] = generate_answer(r)
    print(f"[{r['meta']['type']}] {question}\n  BEFORE: {before[question]}\n  TARGET: {r['messages'][1]['content']}\n")

## 6. Train

About 120 optimizer steps (321 examples × 3 epochs / 8). Expect roughly 60–90 minutes on a T4.

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
train_result = trainer.train()
print(train_result)
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train_pts), label="train loss")
plt.plot(*zip(*eval_pts), marker="o", label="val loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("QLoRA training: Qwen2.5-7B-Instruct on TruthLens RAG data")
plt.savefig("training_curve.png", dpi=120, bbox_inches="tight")
plt.show()
print("best checkpoint:", trainer.state.best_model_checkpoint, "| best val loss:", trainer.state.best_metric)

## 7. After-training generations

In [ ]:
model.eval()
after = {}
for r in probe_rows:
    question = r["messages"][0]["content"].split("QUESTION:")[-1].split("ANSWER:")[0].strip()
    after[question] = generate_answer(r)
    print(f"[{r['meta']['type']}] {question}\n  BEFORE: {before[question]}\n  AFTER:  {after[question]}\n")

## 8. Save and download the adapter

Unzip into `models\lora_adapters\qwen25-7b-truthlens-lora\` locally.

In [ ]:
import shutil

ADAPTER_DIR = f"{OUTPUT_DIR}-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)  # the best checkpoint, loaded at end of training

run_info = {
    "base_model": MODEL_NAME,
    "versions": {"transformers": transformers.__version__, "peft": peft.__version__, "torch": torch.__version__},
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target_modules": TARGET_MODULES},
    "training": {"epochs": EPOCHS, "lr": LEARNING_RATE, "effective_batch": BATCH_SIZE * GRAD_ACCUM, "seed": SEED},
    "data": {"train": len(train_rows), "val": len(val_rows)},
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_val_loss": trainer.state.best_metric,
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "log_history": history,
    "probe_generations": {q: {"before": before[q], "after": after[q]} for q in before},
}
with open(f"{ADAPTER_DIR}/training_run.json", "w") as f:
    json.dump(run_info, f, indent=2)
shutil.copy("training_curve.png", ADAPTER_DIR)

shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
!ls -lh {ADAPTER_DIR} {ADAPTER_DIR}.zip
files.download(f"{ADAPTER_DIR}.zip")